In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("1164").setMaster("local[4]")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/06 01:03:40 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface enp0s3)
25/08/06 01:03:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/06 01:03:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
'''
Table: Products

+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| product_id    | int     |
| new_price     | int     |
| change_date   | date    |
+---------------+---------+
(product_id, change_date) is the primary key (combination of columns with unique values) 
of this table.
Each row of this table indicates that the price of some product was changed 
to a new price at some date.
 

Write a solution to find the prices of all products on 2019-08-16. 
Assume the price of all products before any change is 10.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Products table:
+------------+-----------+-------------+
| product_id | new_price | change_date |
+------------+-----------+-------------+
| 1          | 20        | 2019-08-14  |
| 2          | 50        | 2019-08-14  |
| 1          | 30        | 2019-08-15  |
| 1          | 35        | 2019-08-16  |
| 2          | 65        | 2019-08-17  |
| 3          | 20        | 2019-08-18  |
+------------+-----------+-------------+
Output: 
+------------+-------+
| product_id | price |
+------------+-------+
| 2          | 50    |
| 1          | 35    |
| 3          | 10    |
+------------+-------+
'''

In [2]:
data = [
(1,20,'2019-08-14'),
(2,50,'2019-08-14'),
(1,30,'2019-08-15'),
(1,35,'2019-08-16'),
(2,65,'2019-08-17'),
(3,20,'2019-08-18')
]
schema = ['product_id','new_price','change_date']

In [3]:
df = spark.createDataFrame(data = data, schema = schema)
df.show()

+----------+---------+-----------+
|product_id|new_price|change_date|
+----------+---------+-----------+
|         1|       20| 2019-08-14|
|         2|       50| 2019-08-14|
|         1|       30| 2019-08-15|
|         1|       35| 2019-08-16|
|         2|       65| 2019-08-17|
|         3|       20| 2019-08-18|
+----------+---------+-----------+



In [6]:
temp_df = df.where(F.col("change_date") <= '2019-08-16' )\
            .groupBy(F.col("product_id"))\
            .agg(F.max(F.col("change_date")).alias("change_date"))

temp_df.show()

df1 = temp_df.alias("t").join(df.alias("t1"), 
                   (F.col("t.product_id") == F.col("t1.product_id")) & 
                   (F.col("t.change_date") == F.col("t1.change_date")),
                   'left'
                  )\
             .select(F.col("t.product_id"), F.col("t1.new_price").alias("price"))

df2 = df.alias("t").join(
                         temp_df.alias("t1"), 
                         F.col("t.product_id") == F.col("t1.product_id"),
                         'left_anti'
                        )\
                    .select(F.col("product_id")).distinct()\
                    .withColumn("price",F.lit(10))

df1.union(df2).show()


# 

+----------+-----------+
|product_id|change_date|
+----------+-----------+
|         1| 2019-08-16|
|         2| 2019-08-14|
+----------+-----------+



+----------+-----+
|product_id|price|
+----------+-----+
|         1|   35|
|         2|   50|
|         3|   10|
+----------+-----+



## Melwin here we learn to use of Anitleft Join in place of "not in" 

## SQL Solution

<pre>
WITH TEMP AS (
    SELECT product_id, max(change_date) as change_date
    FROM Products 
    WHERE change_date <= '2019-08-16'   
    GROUP BY product_id
)
SELECT DISTINCT t.product_id, t1.new_price as price
FROM TEMP t  
LEFT JOIN  Products t1 ON t.product_id = t1.product_id 
                        AND t.change_date = t1.change_date
UNION 
SELECT DISTINCT product_id, 10 as price
FROM Products
WHERE product_id not in (SELECT product_id FROM TEMP)
 </pre>